# TRIBE v2 on Kaggle, v2: install once, one click every session after

Runs Meta's **TRIBE v2** on your videos: it predicts cortical responses (20,484 points on fsaverage5, one row per second) and runs **leave-one-modality-out ablations** (no text / no audio / no video).

**What's different from v1**
- **No kernel restart, ever.** TRIBE lives in its own Python environment and runs as a background worker, so the notebook's own Python is never modified.
- **Install once.** That environment is saved in `/kaggle/working/tribe_runtime`. With Kaggle **Persistence** on, it comes back next session, and startup becomes a ~1-minute check instead of a 10–15-minute install. If it's ever lost or broken, the notebook rebuilds it automatically.
- **Both GPUs.** On *GPU T4 x2*, two clips are processed at the same time (one worker per GPU).
- **Checkpointed results.** Finished clips are saved in `/kaggle/working/tribe_outputs` and skipped next time. Adding new videos only processes the new ones.

### One-time setup
1. **Kaggle account phone-verified** (needed for Internet).
2. Accept the Llama 3.2 license at <https://huggingface.co/meta-llama/Llama-3.2-3B> and create a *Read* token at <https://huggingface.co/settings/tokens>.
3. In this notebook, go to **Add-ons ▸ Secrets ▸ Add secret**: Label `HF_TOKEN`, Value = your token, switched **on** for this notebook.
4. Right-hand panel **Session options**: Accelerator **GPU T4 x2**, Internet **On**, and **Persistence: Files** (or *Variables and Files*). Persistence is what saves the environment and results between sessions.
5. Upload your videos as a Kaggle Dataset and attach it with **+ Add Input** (`.mp4 .mov .mkv .avi .webm`). With no videos attached, Meta's 52-second *Sintel* trailer is used as a test.

### Every session
**Run ▸ Run all.** That's it. Download `/kaggle/working/tribe_outputs.zip` from the **Output** panel when it's done.

| | First session | Later sessions (Persistence on) |
|---|---|---|
| Environment | built once (~10–15 min) | checked (~1 min) |
| WhisperX tool + model weights | downloaded (a few min) | downloaded again (a few min; about 20 GB is too big to keep in the 20 GB `/kaggle/working`) |
| Clips already finished | – | skipped |

Kaggle describes persistence as *best-effort*. If the saved environment doesn't come back, this notebook just rebuilds it. Persistence only applies to interactive sessions; a *Save & Run All* version builds in `/tmp` instead.

## 1 · Settings

In [22]:
from pathlib import Path
import os

INPUT_DIR = Path("/kaggle/input")      # attached datasets are searched recursively for videos
MAX_CLIPS = None                       # e.g. 2 to process only the first 2 videos; None = all
MAX_CLIP_SECONDS = 25                  # longer videos are trimmed; MUST stay below 90 s (word-timestamp bug)
USE_SAMPLE_IF_NO_VIDEOS = True         # no videos attached -> Meta's 52 s Sintel demo trailer
ALLOW_NO_TEXT_FALLBACK = True          # if speech transcription fails, continue with video + audio only

# Prediction variants: name -> event types removed before predicting (a removed modality is zero-filled).
VARIANTS = {
    "full":     set(),
    "no_text":  {"Word", "Sentence", "Text"},
    "no_audio": {"Audio"},
    "no_video": {"Video"},
}
# HCP-MMP1 regions for time-course plots: V1 visual, MT motion, FFC faces, A1 auditory,
# STSdp speech/social, 44 + 45 Broca's area
ROIS_TO_PLOT = ["V1", "MT", "FFC", "A1", "STSdp", "44", "45"]

NUM_GPUS = "auto"                      # "auto" = one TRIBE worker per GPU (T4 x2 -> 2 clips at once); 1 = one at a time
STAGGER_SECONDS = 60                   # delay between starting workers so they don't load large models at the same moment
SURFACE_MAPS_FOR_FIRST_N_CLIPS = 3     # cortical surface figures are drawn for this many clips

# Environment pins (changing any of these triggers an automatic rebuild of the saved environment)
PYTHON_VERSION = "3.12"
TRIBEV2_COMMIT = "af58661791a351a448a489042a28f6c37e1c14b7"   # facebookresearch/tribev2 main, 2026-06-23
TRANSFORMERS_SPEC = "transformers>=4.53,<5"                     # V-JEPA2 needs >=4.53; v5 has a known regression

IS_KAGGLE = Path("/kaggle").exists()
BATCH_RUN = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch"   # True inside "Save & Run All" versions
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()
RUNTIME_DIR = (Path("/tmp") if BATCH_RUN else WORK_DIR) / "tribe_runtime"   # saved environment (~7 GB)
OUTPUT_DIR = WORK_DIR / "tribe_outputs"   # results (checkpointed per clip)
CLIP_DIR = Path("/tmp/tribe_clips")       # writable copies of your videos (+ .wav/.tsv side files)
CACHE_DIR = Path("/tmp/tribe_cache")      # TRIBE feature cache (large; not kept)
SAMPLE_URL = "https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4"
VIDEO_SUFFIXES = {".mp4", ".mov", ".mkv", ".avi", ".webm"}

assert 0 < MAX_CLIP_SECONDS < 90, "MAX_CLIP_SECONDS must be below 90 s"
assert VARIANTS.get("full") == set(), "VARIANTS must contain 'full': set()"
print("Environment folder:", RUNTIME_DIR)
print("Results folder:    ", OUTPUT_DIR)

Environment folder: /kaggle/working/tribe_runtime
Results folder:     /kaggle/working/tribe_outputs


## 2 · Hugging Face token
Reads the `HF_TOKEN` Kaggle secret. Access to the gated Llama model is verified in step 4.

In [14]:
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("Could not read the Kaggle secret 'HF_TOKEN':", repr(e))
token = token or os.environ.get("HF_TOKEN")
if not token:
    raise RuntimeError("No Hugging Face token. Add-ons > Secrets > add 'HF_TOKEN' and switch it on for this notebook.")
os.environ["HF_TOKEN"] = token
print("Hugging Face token loaded.")

Hugging Face token loaded.


## 3 · The TRIBE worker script
This is the code that runs inside the saved environment: it loads TRIBE v2, prepares clips, predicts, exports the atlas and draws surface maps. It's written to `tribe_runtime/tribe_runner.py` each session. You normally don't need to touch it.

In [15]:
RUNNER_SRC = r'''# TRIBE v2 worker script - written by the notebook, runs inside the saved TRIBE environment.
# Usage:  python tribe_runner.py <mode> <job.json>
#   check    verify imports, GPU and Hugging Face access
#   prepare  copy/trim videos into a writable folder
#   predict  run TRIBE v2 (full + ablations) on a list of clips; one process per GPU
#   rois     export the HCP-MMP1 atlas (region -> fsaverage5 vertices)
#   maps     draw cortical surface maps for finished clips
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

MODALITY_TYPES = {"Video", "Audio", "Word"}   # event types that feed a TRIBE feature extractor
ABLATION_TO_MODALITY = {"no_text": "text", "no_audio": "audio", "no_video": "video"}


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def write_json(path, obj):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, default=str))
    tmp.replace(path)


# --------------------------------------------------------------------------- check
def mode_check(job):
    import importlib.metadata as md
    import torch
    out = {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "cuda": bool(torch.cuda.is_available()),
        "gpu_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
        "versions": {p: md.version(p) for p in ["numpy", "exca", "neuralset", "transformers"]},
        "ffmpeg": shutil.which("ffmpeg"),
        "uvx": shutil.which("uvx"),
    }
    out["gpus"] = [torch.cuda.get_device_properties(i).name for i in range(out["gpu_count"])]
    from huggingface_hub import hf_hub_download
    try:
        hf_hub_download("meta-llama/Llama-3.2-3B", "config.json")
        out["llama_access"] = True
    except Exception as e:
        out["llama_access"] = False
        out["llama_error"] = f"{type(e).__name__}: {e}"[:600]
    hf_hub_download("facebook/tribev2", "config.yaml")
    import tribev2.demo_utils  # noqa: F401  (full import check: exca / neuralset / torch compatibility)
    write_json(job["result"], out)
    log("check done")


# --------------------------------------------------------------------------- prepare
def ffmpeg_exe():
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    import imageio_ffmpeg
    return imageio_ffmpeg.get_ffmpeg_exe()


def video_duration(path):
    from moviepy import VideoFileClip
    with VideoFileClip(str(path)) as clip:
        return float(clip.duration)


def safe_stem(stem):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", stem).strip("_")[:50] or "clip"


def prepare_clip(src, clip_dir, max_seconds):
    src = Path(src)
    # stable name: same source file -> same name in every session (so finished clips are recognised)
    name = f"{safe_stem(src.stem)}_{hashlib.sha1(str(src).encode()).hexdigest()[:6]}"
    folder = Path(clip_dir) / name
    folder.mkdir(parents=True, exist_ok=True)
    orig = video_duration(src)
    trimmed = orig > max_seconds
    dst = folder / (f"{name}.mp4" if trimmed else f"{name}{src.suffix.lower()}")
    if not dst.exists():
        tmp = folder / f"_partial{dst.suffix}"
        if trimmed:
            subprocess.run([
                ffmpeg_exe(), "-y", "-loglevel", "error", "-i", str(src), "-t", str(max_seconds),
                "-map", "0:v:0", "-map", "0:a:0?", "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
                "-c:v", "libx264", "-preset", "veryfast", "-crf", "18", "-pix_fmt", "yuv420p",
                "-c:a", "aac", "-b:a", "192k", str(tmp),
            ], check=True)
        else:
            shutil.copy2(src, tmp)
        tmp.rename(dst)
    dur = video_duration(dst)
    if dur >= 90:
        raise ValueError(f"{dst.name} is {dur:.1f} s after preparation; must be < 90 s")
    return {"name": name, "source": str(src), "path": str(dst), "orig_duration": round(orig, 2),
            "duration": round(dur, 2), "trimmed": trimmed}


def mode_prepare(job):
    clips, errors = [], []
    for src in job["sources"]:
        try:
            clips.append(prepare_clip(src, job["clip_dir"], float(job["max_clip_seconds"])))
        except Exception as e:
            errors.append({"source": src, "error": f"{type(e).__name__}: {e}"})
            log(f"SKIPPING {src}: {type(e).__name__}: {e}")
    names = [c["name"] for c in clips]
    if len(names) != len(set(names)):
        raise ValueError(f"Duplicate clip names: {names}")
    write_json(job["result"], {"clips": clips, "errors": errors})
    log(f"prepared {len(clips)} clip(s), {len(errors)} skipped")


# --------------------------------------------------------------------------- predict
def build_events(model, clip, allow_no_text):
    """TRIBE's own events pipeline (audio extraction + WhisperX words). Falls back to no text if it fails."""
    import pandas as pd
    from tribev2.demo_utils import get_audio_and_text_events
    try:
        return model.get_events_dataframe(video_path=clip["path"]).reset_index(drop=True), None
    except Exception as e:
        if not allow_no_text:
            raise
        reason = f"{type(e).__name__}: {str(e)[-800:]}"
        log(f"  !! text pipeline failed -> continuing WITHOUT the text modality: {reason}")
        event = {"type": "Video", "filepath": clip["path"], "start": 0,
                 "timeline": "default", "subject": "default"}
        return get_audio_and_text_events(pd.DataFrame([event]), audio_only=True).reset_index(drop=True), reason


def validate_events(df, clip):
    """Guard against the long-clip timestamp bug and other silent pipeline errors."""
    types = set(df["type"].unique())
    if "Video" not in types:
        raise ValueError(f"No Video event produced (event types: {sorted(types)})")
    n_video = int((df["type"] == "Video").sum())
    if n_video != 1:
        raise ValueError(f"Expected 1 Video event for a < 90 s clip, got {n_video} (clip was chunked)")
    words = df[df["type"] == "Word"]
    if len(words):
        lo = float(words["start"].min())
        hi = float((words["start"] + words["duration"].fillna(0)).max())
        if lo < -1.0 or hi > clip["duration"] + 1.0:
            raise ValueError(f"Word timestamps [{lo:.1f}, {hi:.1f}] s fall outside the {clip['duration']:.1f} s clip")
        dups = int(words.duplicated(subset=["start", "text"]).sum())
        if dups:
            log(f"  warning: {dups} words share an identical start time and text")
    return {t: int((df["type"] == t).sum()) for t in sorted(types)}


def predict_variant(model, df, drop_types):
    """Predict with some event types removed. Returns (preds [T, 20484], times [T] in seconds)."""
    import numpy as np
    sub = df.loc[~df["type"].isin(drop_types)].copy().reset_index(drop=True)
    preds, segments = model.predict(events=sub, verbose=False)
    times = np.array([float(s.start) for s in segments])   # rows are NOT guaranteed to be 0,1,2,... s
    if preds.shape[0] != len(times):
        raise ValueError(f"{preds.shape[0]} prediction rows vs {len(times)} segments")
    return preds.astype(np.float32), times


def mode_predict(job):
    import numpy as np
    import torch
    from tribev2.demo_utils import TribeModel

    out_root = Path(job["out_dir"]) / "clips"
    variants = {k: set(v) for k, v in job["variants"].items()}
    gpu = torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else "CPU"
    log(f"worker {job['worker']} on {gpu}: {len(job['clips'])} clip(s) in the shared queue")
    t0 = time.time()
    model = TribeModel.from_pretrained("facebook/tribev2", cache_folder=job["cache_dir"],
                                       config_update={"data.num_workers": 0})
    log(f"model loaded in {time.time() - t0:.0f} s")

    claim_dir = Path(job["claim_dir"])
    claim_dir.mkdir(parents=True, exist_ok=True)
    n_done = 0
    for clip in job["clips"]:          # shared queue: every worker gets the same list and claims clips one by one
        try:
            os.close(os.open(claim_dir / f"{clip['name']}.claim", os.O_CREAT | os.O_EXCL | os.O_WRONLY))
        except FileExistsError:
            continue                   # another worker took this clip
        out = out_root / clip["name"]
        out.mkdir(parents=True, exist_ok=True)
        info_path = out / "info.json"
        if info_path.exists() and json.loads(info_path.read_text()).get("status") == "done":
            log(f"{clip['name']}: already done, skipping")
            continue
        n_done += 1
        log(f"clip #{n_done}: {clip['name']} ({clip['duration']} s)")
        info = dict(clip, status="running", worker=job["worker"], gpu=gpu, variants={}, skipped_variants={})
        torch.cuda.reset_peak_memory_stats()
        t_clip = time.time()
        try:
            t = time.time()
            events, text_error = build_events(model, clip, job["allow_no_text"])
            info["text_error"] = text_error
            info["event_counts"] = validate_events(events, clip)
            info["events_seconds"] = round(time.time() - t, 1)
            events.to_csv(out / "events.csv", index=False)
            log(f"  events {info['event_counts']} ({info['events_seconds']} s)")

            present = set(events["type"])
            for vname, drop in variants.items():
                if drop and not (present & drop):
                    info["skipped_variants"][vname] = "modality absent in this clip (identical to full)"
                    log(f"  {vname:9s} skipped: modality absent in this clip")
                    continue
                if not (present - drop) & MODALITY_TYPES:
                    info["skipped_variants"][vname] = "no other modality left to predict from"
                    log(f"  {vname:9s} skipped: no other modality left to predict from")
                    continue
                t = time.time()
                preds, times = predict_variant(model, events, drop)
                np.save(out / f"preds_{vname}.npy", preds)
                np.save(out / f"times_{vname}.npy", times)
                info["variants"][vname] = {"shape": list(preds.shape), "seconds": round(time.time() - t, 1)}
                log(f"  {vname:9s} {tuple(preds.shape)} in {time.time() - t:.0f} s")
            info["status"] = "done"
        except Exception as e:
            info["status"] = "failed"
            info["error"] = f"{type(e).__name__}: {e}"
            traceback.print_exc()
            sys.stdout.flush()
        info["total_seconds"] = round(time.time() - t_clip, 1)
        info["peak_gpu_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else 0
        write_json(info_path, info)
        torch.cuda.empty_cache()
        log(f"  -> {info['status']} in {info['total_seconds']:.0f} s, peak GPU {info['peak_gpu_gb']} GB")
    log(f"worker finished ({n_done} clip(s) processed)")


# --------------------------------------------------------------------------- rois
def mode_rois(job):
    import numpy as np
    from tribev2.utils import get_hcp_labels
    hcp = get_hcp_labels(mesh="fsaverage5", combine=False, hemi="both")
    names = [r for r in hcp if any(c.isalnum() for c in r)]
    idx = [np.asarray(hcp[r], dtype=np.int64) for r in names]
    result = Path(job["result"])
    tmp = result.with_name("tmp_" + result.name)
    np.savez(tmp, names=np.array(names), indices=np.concatenate(idx),
             offsets=np.cumsum([0] + [len(i) for i in idx]))
    tmp.replace(result)
    log(f"saved {len(names)} HCP regions to {result}")


# --------------------------------------------------------------------------- maps
def mode_maps(job):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np
    from tribev2.plotting import PlotBrainNilearn

    plotter = PlotBrainNilearn(mesh="fsaverage5")
    made = []
    for name in job["clips"]:
        folder = Path(job["out_dir"]) / "clips" / name
        full = np.load(folder / "preds_full.npy")
        maps = {"mean activity (full)": full.mean(axis=0)}
        for vname, modality in ABLATION_TO_MODALITY.items():
            p = folder / f"preds_{vname}.npy"
            if p.exists():
                abl = np.load(p)
                if abl.shape == full.shape:
                    maps[f"{modality} effect (RMS)"] = np.sqrt(((full - abl) ** 2).mean(axis=0))
        for title, data in maps.items():
            try:
                data = np.asarray(data, dtype=np.float64)
                if title.startswith("mean"):
                    vmax = float(np.percentile(np.abs(data), 99)) or 1.0
                    plotter.plot_surf(data, views=["left", "right"], cmap=matplotlib.colormaps["RdBu_r"],
                                      vmin=-vmax, vmax=vmax, colorbar=True, colorbar_title="predicted BOLD")
                else:
                    vmax = float(np.percentile(data, 99)) or 1.0
                    plotter.plot_surf(data, views=["left", "right"], cmap=matplotlib.colormaps["Blues"],
                                      vmin=0.0, vmax=vmax, colorbar=True, colorbar_title="RMS change")
                fig = plt.gcf()
                fig.suptitle(f"{name}: {title}")
                fname = re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")
                path = folder / f"surface_{fname}.png"
                fig.savefig(path, bbox_inches="tight", dpi=130)
                made.append(str(path))
            except Exception as e:
                log(f"  surface map '{title}' for {name} failed: {type(e).__name__}: {e}")
            finally:
                plt.close("all")
    write_json(job["result"], {"images": made})
    log(f"saved {len(made)} surface map(s)")


MODES = {"check": mode_check, "prepare": mode_prepare, "predict": mode_predict,
         "rois": mode_rois, "maps": mode_maps}

if __name__ == "__main__":
    mode, job_path = sys.argv[1], sys.argv[2]
    MODES[mode](json.loads(Path(job_path).read_text()))
'''
print(f"Worker script: {len(RUNNER_SRC.splitlines())} lines")

Worker script: 308 lines


## 4 · Load (or build) the saved TRIBE environment
This checks the environment saved in `tribe_runtime/`. If it's missing, broken or out of date, it's rebuilt with [`uv`](https://docs.astral.sh/uv/) (its own Python 3.12 plus pinned packages: `tribev2` at a fixed commit, `exca==0.5.20`, `torch==2.6.0`, `numpy==2.2.6`, `transformers<5`). Then it prepares WhisperX, which TRIBE uses to transcribe speech, and checks the GPU and Hugging Face access.

In [23]:
import hashlib, json, shutil, subprocess, sys, sysconfig, time

REQUIREMENTS = [
    f"tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git@{TRIBEV2_COMMIT}",
    "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "numpy==2.2.6",
    "exca==0.5.20", "neuralset==0.0.2", "neuraltrain==0.0.2", TRANSFORMERS_SPEC,
    "uv",   # provides `uvx`, which tribev2 uses to launch WhisperX
]
SPEC_ID = hashlib.sha1(json.dumps([PYTHON_VERSION, *REQUIREMENTS]).encode()).hexdigest()[:12]
PY_DIR = RUNTIME_DIR / "python"
ENV_DIR = RUNTIME_DIR / "env"
ENV_PY = ENV_DIR / "bin" / "python"
MARKER = ENV_DIR / "tribe_ready.json"
RUNNER = RUNTIME_DIR / "tribe_runner.py"
JOBS_DIR = RUNTIME_DIR / "jobs"
LOG_DIR = OUTPUT_DIR / "logs"
CLIPS_OUT = OUTPUT_DIR / "clips"


def runtime_env():
    """Environment variables for everything that runs inside the TRIBE environment."""
    drop = ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "UV_MANAGED_PYTHON", "UV_NO_MANAGED_PYTHON")
    env = {k: v for k, v in os.environ.items() if k not in drop}
    env.update({
        "UV_PYTHON_INSTALL_DIR": str(PY_DIR), "UV_PYTHON_PREFERENCE": "only-managed",
        "UV_CACHE_DIR": "/tmp/uv_cache", "UV_LINK_MODE": "copy",
        "PATH": str(ENV_DIR / "bin") + os.pathsep + os.environ.get("PATH", ""),
        "MPLBACKEND": "Agg", "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1",
        "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",   # NEW: reduces GPU memory fragmentation
    })
    return env


def run(cmd, env=None, fatal=True, quiet=False):
    """Run a command and stream its output (quiet: only show the tail if it fails)."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    tail = []
    for line in proc.stdout:
        if quiet:
            tail = (tail + [line])[-25:]
        else:
            print(line, end="", flush=True)
    rc = proc.wait()
    if rc != 0:
        if quiet:
            print("".join(tail), end="")
        msg = f"Command failed with exit code {rc}: {' '.join(cmd)}"
        if fatal:
            raise RuntimeError(msg)
        print("WARNING:", msg)
    return rc == 0


def import_check():
    r = subprocess.run([str(ENV_PY), "-c", "import tribev2.demo_utils"], env=runtime_env(),
                       capture_output=True, text=True)
    return r.returncode == 0, ((r.stderr or "").strip().splitlines() or [""])[-1][:300]


def runtime_status():
    if not ENV_PY.exists():
        return "not built yet"
    try:
        spec = json.loads(MARKER.read_text()).get("spec")
    except Exception:
        return "incomplete (a previous build did not finish)"
    if spec != SPEC_ID:
        return "outdated (package settings changed)"
    ok, err = import_check()
    return "ok" if ok else f"broken ({err})"


def get_uv():
    uv = shutil.which("uv")
    if uv:
        return uv
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
    uv = shutil.which("uv") or str(Path(sysconfig.get_path("scripts")) / "uv")
    if not Path(uv).exists():
        raise RuntimeError("Could not install uv")
    return uv


def build_runtime():
    shutil.rmtree(ENV_DIR, ignore_errors=True)
    RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
    uv, env = get_uv(), runtime_env()
    run([uv, "python", "install", PYTHON_VERSION], env=env)
    run([uv, "venv", ENV_DIR, "--python", PYTHON_VERSION], env=env)
    run([uv, "pip", "install", "--python", ENV_PY, *REQUIREMENTS], env=env)
    run([ENV_PY, "-m", "spacy", "download", "en_core_web_lg"], env=env, fatal=False, quiet=True)
    ok, err = import_check()
    if not ok:
        raise RuntimeError(f"Environment was built but `import tribev2` fails: {err}")
    MARKER.write_text(json.dumps({"spec": SPEC_ID, "requirements": REQUIREMENTS,
                                  "built": time.strftime("%Y-%m-%d %H:%M")}))


def run_worker(mode, job, tag):
    """Run tribe_runner.py <mode> in the TRIBE environment and return its result."""
    job = dict(job)
    job.setdefault("result", str(JOBS_DIR / f"{tag}_result.json"))
    job_path = JOBS_DIR / f"{tag}.json"
    job_path.write_text(json.dumps(job, default=str))
    run([ENV_PY, RUNNER, mode, job_path], env=runtime_env())
    result = Path(job["result"])
    return json.loads(result.read_text()) if result.suffix == ".json" else result


def dir_size_gb(path):
    r = subprocess.run(["du", "-sb", str(path)], capture_output=True, text=True)
    return int(r.stdout.split()[0]) / 1e9 if r.returncode == 0 and r.stdout.strip() else 0.0


# ---- load or build ---------------------------------------------------------
for d in (RUNTIME_DIR, JOBS_DIR, LOG_DIR, CLIPS_OUT, CLIP_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
RUNNER.write_text(RUNNER_SRC)

t0 = time.time()
status = runtime_status()
print(f"Saved TRIBE environment: {status}")
if status != "ok":
    print("-> Building it now (first session, or it was lost/changed). This takes ~10-15 min, once.\n")
    build_runtime()
print(f"Environment ready after {time.time() - t0:.0f} s ({dir_size_gb(RUNTIME_DIR):.1f} GB in {RUNTIME_DIR})")

# ---- ffmpeg on PATH (WhisperX needs it) -------------------------------------
link = ENV_DIR / "bin" / "ffmpeg"
if shutil.which("ffmpeg") is None and not link.exists():
    if link.is_symlink():
        link.unlink()                         # dangling link from an older session
    exe = subprocess.run([str(ENV_PY), "-c", "import imageio_ffmpeg; print(imageio_ffmpeg.get_ffmpeg_exe())"],
                         env=runtime_env(), capture_output=True, text=True, check=True).stdout.strip()
    link.symlink_to(exe)
    print("Using the ffmpeg bundled with imageio-ffmpeg")

# ---- WhisperX (tribev2 launches it with `uvx whisperx`; its tool environment is rebuilt each session) ----
t1 = time.time()
if run([ENV_DIR / "bin" / "uvx", "whisperx", "--help"], env=runtime_env(), fatal=False, quiet=True):
    print(f"WhisperX ready ({time.time() - t1:.0f} s)")
else:
    print("WARNING: WhisperX could not start; clips will run without the text modality.")

# ---- GPU + Hugging Face access check (runs inside the TRIBE environment) ------
# CHECK = run_worker("check", {}, "check")
# print(f"\nPython {CHECK['python']} | torch {CHECK['torch']} | " +
#       " | ".join(f"{k} {v}" for k, v in CHECK["versions"].items()))
# print("GPUs:", CHECK["gpus"] or "none")
# if not CHECK["cuda"]:
#     raise RuntimeError("No GPU visible. Session options > Accelerator > GPU T4 x2.")
# if not CHECK["llama_access"]:
#     raise RuntimeError("Your token cannot download meta-llama/Llama-3.2-3B. Accept the license at "
#                        "https://huggingface.co/meta-llama/Llama-3.2-3B and wait for approval.\n"
#                        + CHECK.get("llama_error", ""))
# N_WORKERS_MAX = CHECK["gpu_count"] if NUM_GPUS == "auto" else max(1, min(int(NUM_GPUS), CHECK["gpu_count"]))
# print(f"Hugging Face access OK. Up to {N_WORKERS_MAX} clip(s) will run at the same time.")

# ---- Use the ungated Llama-3.2-3B mirror (identical weights) ------------------
# ---- Use the ungated Llama-3.2-3B mirror (identical weights) ------------------
GATED, MIRROR = "meta-llama/Llama-3.2-3B", "unsloth/Llama-3.2-3B"

def env_out(code):
    return subprocess.run([str(ENV_PY), "-c", code], env=runtime_env(),
                          capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1]

tribe_cfg = env_out("from huggingface_hub import snapshot_download; "
                    "print(snapshot_download('facebook/tribev2', allow_patterns=['*.yaml','*.yml','*.json']))")
roots = [Path(env_out("import tribev2, os; print(os.path.dirname(tribev2.__file__))")),
         Path(env_out("import neuralset, os; print(os.path.dirname(neuralset.__file__))")),
         Path(tribe_cfg), CACHE_DIR, RUNNER]

patched = []
for root in roots:
    files = [root] if root.is_file() else [p for p in root.rglob("*")
             if p.suffix in {".py", ".yaml", ".yml", ".json"} and p.is_file()]
    for f in files:
        try:
            txt = f.read_text()
        except Exception:
            continue
        if GATED in txt:
            f.write_text(txt.replace(GATED, MIRROR))
            patched.append(str(f))
print(f"Swapped {GATED} -> {MIRROR} in {len(patched)} file(s):", *patched, sep="\n  ")

# show how TRIBE's config names the text model (for debugging)
for f in Path(tribe_cfg).rglob("*.yaml"):
    for i, line in enumerate(f.read_text().splitlines()):
        if "lama" in line.lower():
            print(f"  {f.name}:{i+1}: {line.strip()}")

# ---- GPU + Hugging Face access check (runs inside the TRIBE environment) ------
CHECK = run_worker("check", {}, "check")
print(f"\nPython {CHECK['python']} | torch {CHECK['torch']} | " +
      " | ".join(f"{k} {v}" for k, v in CHECK["versions"].items()))
print("GPUs:", CHECK["gpus"] or "none")
if not CHECK["cuda"]:
    raise RuntimeError("No GPU visible. Session options > Accelerator > GPU T4 x2.")
if not CHECK["llama_access"]:
    raise RuntimeError("Your token cannot download the Llama model.\n" + CHECK.get("llama_error", ""))
N_WORKERS_MAX = CHECK["gpu_count"] if NUM_GPUS == "auto" else max(1, min(int(NUM_GPUS), CHECK["gpu_count"]))
print(f"Hugging Face access OK. Up to {N_WORKERS_MAX} clip(s) will run at the same time.")

Saved TRIBE environment: ok
Environment ready after 10 s (7.9 GB in /kaggle/working/tribe_runtime)
$ /kaggle/working/tribe_runtime/env/bin/uvx whisperx --help
WhisperX ready (3 s)
Swapped meta-llama/Llama-3.2-3B -> unsloth/Llama-3.2-3B in 1 file(s):
  /kaggle/working/tribe_runtime/tribe_runner.py
  config.yaml:194: model_name: unsloth/Llama-3.2-3B
$ /kaggle/working/tribe_runtime/env/bin/python /kaggle/working/tribe_runtime/tribe_runner.py check /kaggle/working/tribe_runtime/jobs/check.json
/kaggle/working/tribe_runtime/env/lib/python3.12/site-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-09-29 08:03:25 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_miss

## 5 · Find and prepare your clips
Videos are copied into a writable folder (`/kaggle/input` is read-only, and TRIBE writes audio files next to each video). Anything longer than `MAX_CLIP_SECONDS` is trimmed. Each clip gets a stable name, so finished clips are recognised in later sessions.

In [32]:
!rm -rf /tmp/tribe_clips

In [33]:
import shutil
import urllib.request
import pandas as pd
from IPython.display import display

sources = []
if INPUT_DIR.exists():
    sources = sorted(str(p) for p in INPUT_DIR.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_SUFFIXES)
if not sources:
    if not USE_SAMPLE_IF_NO_VIDEOS:
        raise RuntimeError(f"No videos found under {INPUT_DIR}. Attach a dataset with '+ Add Input'.")
    sample = Path("/tmp/tribe_samples/sintel_trailer.mp4")
    if not sample.exists() or sample.stat().st_size == 0:
        print("No videos found -> downloading the Sintel demo trailer.")
        sample.parent.mkdir(parents=True, exist_ok=True)
        req_headers = {"User-Agent": "Mozilla/5.0"}
        for url in (SAMPLE_URL, "https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4"):
            try:
                req = urllib.request.Request(url, headers=req_headers)
                with urllib.request.urlopen(req, timeout=60) as r, open(sample, "wb") as f:
                    shutil.copyfileobj(r, f)
                print("Downloaded sample from", url)
                break
            except Exception as e:
                print("Failed:", url, "-", e)
                sample.unlink(missing_ok=True)
        else:
            raise RuntimeError("Could not download a sample video. Attach your own with '+ Add Input'.")
    sources = [str(sample)]
if MAX_CLIPS is not None:
    sources = sources[:MAX_CLIPS]

prepared = run_worker("prepare", {"sources": sources, "clip_dir": str(CLIP_DIR),
                                  "max_clip_seconds": MAX_CLIP_SECONDS}, "prepare")
clips = prepared["clips"]
for err in prepared["errors"]:
    print("SKIPPED:", err["source"], "-", err["error"])
if not clips:
    raise RuntimeError("No usable clips.")
display(pd.DataFrame([{k: c[k] for k in ("name", "orig_duration", "duration", "trimmed")} for c in clips]))

$ /kaggle/working/tribe_runtime/env/bin/python /kaggle/working/tribe_runtime/tribe_runner.py prepare /kaggle/working/tribe_runtime/jobs/prepare.json
[08:06:04] prepared 1 clip(s), 0 skipped


,name,orig_duration,duration,trimmed
0,sintel_trailer_38c660,52.21,25.0,True


## 6 · Run TRIBE v2 (one worker per GPU)
Clips already finished in an earlier session are skipped. The rest go into one shared queue, longest first: each GPU's worker takes the next unclaimed clip until none are left, so neither GPU sits idle. Each worker's log is streamed below with a `[gpu0]` / `[gpu1]` prefix and also saved in `tribe_outputs/logs/`. If a worker dies (for example if it runs out of memory), its unfinished clips are retried automatically on a single GPU.

Rows of each prediction are time-aligned with the stimulus: TRIBE has already shifted its output by 5 s for the hemodynamic delay. Do **not** add another lag.

In [ ]:
def clip_status(name):
    try:
        return json.loads((CLIPS_OUT / name / "info.json").read_text()).get("status")
    except Exception:
        return None


def _tail(procs):
    for x in procs:
        with open(x["log"], "rb") as f:
            f.seek(x["pos"])
            data = f.read()
        if x["p"].poll() is None:                 # still running: only print complete lines
            data = data[: data.rfind(b"\n") + 1]
        x["pos"] += len(data)
        for line in data.decode("utf-8", errors="replace").splitlines():
            print(f"[{x['tag']}] {line}", flush=True)


def launch(queue, n_workers, label):
    """Start n_workers (one per GPU). They share one queue (longest clips first) and claim clips one by one."""
    queue = sorted(queue, key=lambda c: -c["duration"])
    claim_dir = JOBS_DIR / f"claims_{label}_{time.strftime('%Y%m%d-%H%M%S')}"
    procs = []
    try:
        for i in range(n_workers):
            job = {"worker": f"gpu{i}", "clips": queue, "claim_dir": str(claim_dir), "out_dir": str(OUTPUT_DIR),
                   "variants": {k: sorted(v) for k, v in VARIANTS.items()},
                   "cache_dir": str(CACHE_DIR / f"gpu{i}"), "allow_no_text": ALLOW_NO_TEXT_FALLBACK}
            job_path = JOBS_DIR / f"predict_{label}_gpu{i}.json"
            job_path.write_text(json.dumps(job))
            log_path = LOG_DIR / f"{time.strftime('%Y%m%d-%H%M%S')}_{label}_gpu{i}.log"
            env = runtime_env()
            env["CUDA_VISIBLE_DEVICES"] = str(i)
            fh = open(log_path, "wb")
            p = subprocess.Popen([str(ENV_PY), str(RUNNER), "predict", str(job_path)],
                                 stdout=fh, stderr=subprocess.STDOUT, env=env)
            procs.append({"tag": f"gpu{i}", "p": p, "fh": fh, "log": log_path, "pos": 0})
            print(f"Started worker gpu{i}; log: logs/{log_path.name}", flush=True)
            if i < n_workers - 1:
                t_end = time.time() + STAGGER_SECONDS
                while time.time() < t_end and p.poll() is None:
                    _tail(procs)
                    time.sleep(2)
        while any(x["p"].poll() is None for x in procs):
            _tail(procs)
            time.sleep(3)
        _tail(procs)
    except KeyboardInterrupt:
        for x in procs:
            if x["p"].poll() is None:
                x["p"].terminate()
        raise
    finally:
        for x in procs:
            x["fh"].close()
    return {x["tag"]: x["p"].returncode for x in procs}


t_run = time.time()
pending = [c for c in clips if clip_status(c["name"]) != "done"]
print(f"{len(clips) - len(pending)} clip(s) already done in earlier sessions; {len(pending)} to run.\n")
if pending:
    n_workers = max(1, min(N_WORKERS_MAX, len(pending)))
    codes = launch(pending, n_workers, "run")
    print("\nWorker exit codes:", codes)
    left = [c for c in pending if clip_status(c["name"]) != "done"]
    if left and n_workers > 1:
        print(f"\n{len(left)} clip(s) did not finish -> retrying them one at a time on a single GPU\n")
        print("Retry exit code:", launch(left, 1, "retry"))

results = {}
for c in clips:
    p = CLIPS_OUT / c["name"] / "info.json"
    if p.exists():
        results[c["name"]] = json.loads(p.read_text())
done = [n for n, r in results.items() if r.get("status") == "done"]
summary = pd.DataFrame([{
    "clip": n, "status": r.get("status"), "worker": r.get("worker"), "minutes": round(r.get("total_seconds", 0) / 60, 1),
    "peak_gpu_gb": r.get("peak_gpu_gb"), "variants": ", ".join(r.get("variants", {})),
    "note": r.get("error") or ("no text: " + r["text_error"][:80] if r.get("text_error") else ""),
} for n, r in results.items()])
display(summary)
print(f"Run step took {(time.time() - t_run) / 60:.1f} min. Done: {len(done)} / {len(clips)}")
if not done:
    raise RuntimeError("No clip finished successfully - see the worker logs above.")

0 clip(s) already done in earlier sessions; 1 to run.

Started worker gpu0; log: logs/20260929-080609_run_gpu0.log
[gpu0] /kaggle/working/tribe_runtime/env/lib/python3.12/site-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
[gpu0]   warnings.warn(
[gpu0] 2026-09-29 08:06:17 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
[gpu0] [08:06:17] worker gpu0 on Tesla T4: 1 clip(s) in the shared queue
[gpu0] 2026-09-29 08:06:17 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is 

## 7 · Region-level analysis (HCP-MMP1 atlas, 180 regions)
The very first time, the atlas is exported from the TRIBE environment via MNE (a ~1.6 GB one-time download). It is saved to `tribe_outputs/`, so later sessions skip this. The analysis itself runs in this notebook with plain NumPy and pandas.

Saved per clip:
- `roi_timecourses_<variant>.csv`: time × 180 regions, both hemispheres averaged
- `modality_dependence.csv`: RMS change in each region's prediction when a modality is removed. Higher means the region relies more on that modality.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

HCP_NPZ = OUTPUT_DIR / "hcp_rois_fsaverage5.npz"
if not HCP_NPZ.exists():
    print("First time only: exporting the HCP-MMP1 atlas (downloads ~1.6 GB via MNE)...")
    run_worker("rois", {"result": str(HCP_NPZ)}, "rois")
_z = np.load(HCP_NPZ)
ROI_NAMES = [str(n) for n in _z["names"]]
ROI_IDX = [_z["indices"][_z["offsets"][i]:_z["offsets"][i + 1]] for i in range(len(ROI_NAMES))]
print(f"{len(ROI_NAMES)} HCP regions")
missing = [r for r in ROIS_TO_PLOT if r not in ROI_NAMES]
if missing:
    print("Not found in atlas (ignored):", missing)
PLOT_ROIS = [r for r in ROIS_TO_PLOT if r in ROI_NAMES]
ABLATION_TO_MODALITY = {"no_text": "text", "no_audio": "audio", "no_video": "video"}

# plot style: one fixed colour per modality (validated categorical slots 1-3)
MODALITY_COLORS = {"text": "#2a78d6", "audio": "#eb6834", "video": "#1baf7a"}
INK, MUTED = "#0b0b0b", "#52514e"
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": MUTED,
    "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "figure.dpi": 110,
})


def roi_timecourses(preds):
    return np.stack([preds[:, idx].mean(axis=1) for idx in ROI_IDX], axis=1)


def load_variant(folder, vname):
    p = folder / f"preds_{vname}.npy"
    if not p.exists():
        return None, None
    return np.load(p), np.load(folder / f"times_{vname}.npy")


dependence_tables = {}
for name in done:
    folder = CLIPS_OUT / name
    full, t_full = load_variant(folder, "full")
    roi_full = roi_timecourses(full)
    df_full = pd.DataFrame(roi_full, columns=ROI_NAMES)
    df_full.insert(0, "time_s", t_full)
    df_full.to_csv(folder / "roi_timecourses_full.csv", index=False)
    dep = {}
    for vname, modality in ABLATION_TO_MODALITY.items():
        abl, t_abl = load_variant(folder, vname)
        if abl is None:
            continue
        if abl.shape != full.shape or not np.allclose(t_abl, t_full):
            print(f"  {name}: {vname} rows do not line up with full ({abl.shape} vs {full.shape}); skipped")
            continue
        roi_abl = roi_timecourses(abl)
        df_abl = pd.DataFrame(roi_abl, columns=ROI_NAMES)
        df_abl.insert(0, "time_s", t_abl)
        df_abl.to_csv(folder / f"roi_timecourses_{vname}.csv", index=False)
        dep[modality] = np.sqrt(((roi_full - roi_abl) ** 2).mean(axis=0))
    dep_df = pd.DataFrame(dep, index=pd.Index(ROI_NAMES, name="roi"))
    dep_df.to_csv(folder / "modality_dependence.csv")
    dependence_tables[name] = dep_df
print(f"Saved ROI time courses and modality dependence for {len(done)} clip(s)")

### 7a · Region time courses (full model), one panel per region

In [ ]:
for name in done:
    df_full = pd.read_csv(CLIPS_OUT / name / "roi_timecourses_full.csv")
    n = len(PLOT_ROIS)
    if n == 0:
        break
    fig, axes = plt.subplots(n, 1, figsize=(9, 1.25 * n + 0.6), sharex=True, squeeze=False)
    for ax, roi in zip(axes[:, 0], PLOT_ROIS):
        ax.plot(df_full["time_s"], df_full[roi], color=INK, lw=2)
        ax.axhline(0, color=MUTED, lw=0.8)
        ax.set_ylabel(roi, rotation=0, ha="right", va="center", color=INK)
    axes[-1, 0].set_xlabel("time in clip (s)")
    fig.suptitle(f"{name}: predicted response by region", x=0.01, ha="left", color=INK)
    fig.tight_layout()
    fig.savefig(CLIPS_OUT / name / "roi_timecourses.png", bbox_inches="tight")
    plt.show()

### 7b · Which regions depend on which modality?
These are the top 10 regions per modality, ranked by how much the prediction changes when that modality is removed. Each panel averages over the clips that contain that modality; the result is saved as `modality_dependence_all_clips.csv`.

In [ ]:
if dependence_tables and any(len(t.columns) for t in dependence_tables.values()):
    avg_dep = pd.concat(dependence_tables.values()).groupby(level=0).mean()
    avg_dep = avg_dep.reindex(index=ROI_NAMES,
                              columns=[m for m in ["text", "audio", "video"] if m in avg_dep.columns])
    avg_dep.to_csv(OUTPUT_DIR / "modality_dependence_all_clips.csv")
    mods = list(avg_dep.columns)
    fig, axes = plt.subplots(1, len(mods), figsize=(4.2 * len(mods), 4.2), squeeze=False)
    for ax, m in zip(axes[0], mods):
        top = avg_dep[m].dropna().sort_values(ascending=False).head(10)[::-1]
        ax.barh(top.index, top.values, color=MODALITY_COLORS[m], height=0.6)
        n_clips = sum(m in t.columns for t in dependence_tables.values())
        ax.set_title(f"remove {m}  (n = {n_clips} clip{'s' if n_clips != 1 else ''})", loc="left", color=INK)
        ax.set_xlabel("RMS change in prediction")
        ax.grid(axis="y", visible=False)
    fig.suptitle("Regions most dependent on each modality (mean over clips that contain it)",
                 x=0.01, ha="left", color=INK)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "modality_dependence_top10.png", bbox_inches="tight")
    plt.show()
    display(avg_dep.sort_values(mods[0], ascending=False).head(15).round(4))
else:
    print("No ablation results available (all ablations were skipped or failed).")

## 8 · Cortical surface maps (best effort)
These show the time-averaged full prediction and each modality's vertex-level RMS effect, drawn inside the TRIBE environment with Nilearn and saved as PNGs. Maps that already exist are reused. If drawing fails, the numbers above are unaffected.

In [ ]:
from IPython.display import Image

map_clips = done[:SURFACE_MAPS_FOR_FIRST_N_CLIPS]
todo = [n for n in map_clips if not (CLIPS_OUT / n / "surface_mean_activity_full.png").exists()]
if todo:
    try:
        run_worker("maps", {"clips": todo, "out_dir": str(OUTPUT_DIR)}, "maps")
    except Exception as e:
        print("Surface maps failed (numeric results are unaffected):", e)
for name in map_clips:
    for png in sorted((CLIPS_OUT / name).glob("surface_*.png")):
        print(png.name)
        display(Image(filename=str(png)))

## 9 · Package the outputs

In [ ]:
manifest = {
    "tribev2_commit": TRIBEV2_COMMIT,
    "environment": json.loads(MARKER.read_text()) if MARKER.exists() else None,
    "check": CHECK,
    "variants": {k: sorted(v) for k, v in VARIANTS.items()},
    "max_clip_seconds": MAX_CLIP_SECONDS,
    "clips": results,
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
archive = shutil.make_archive(str(WORK_DIR / "tribe_outputs"), "zip", OUTPUT_DIR)
print(f"Wrote {archive} ({Path(archive).stat().st_size / 1e6:.1f} MB) - download it from the Output panel.")

used = dir_size_gb(WORK_DIR)
print(f"/kaggle/working uses {used:.1f} GB "
      f"(environment {dir_size_gb(RUNTIME_DIR):.1f} GB, results {dir_size_gb(OUTPUT_DIR):.1f} GB).")
if IS_KAGGLE and used > 17:
    print("WARNING: close to Kaggle's ~20 GB limit for /kaggle/working. Download and delete old results "
          "(tribe_outputs/clips/...) or persistence may fail to save the environment.")

## What's in `tribe_outputs/`
| File | Contents |
|---|---|
| `clips/<clip>/preds_<variant>.npy` | float32 `(T, 20484)`: predicted response per second × fsaverage5 vertex (left hemisphere first) |
| `clips/<clip>/times_<variant>.npy` | time (s) of each row. Use this, not the row index |
| `clips/<clip>/events.csv` | stimulus events TRIBE saw (video, audio, transcribed words) |
| `clips/<clip>/roi_timecourses_<variant>.csv` | time × 180 HCP-MMP1 regions |
| `clips/<clip>/modality_dependence.csv` | per region: RMS change when text / audio / video is removed |
| `clips/<clip>/info.json` | status, timings, which GPU, peak GPU memory, event counts, errors |
| `modality_dependence_all_clips.csv`, `*.png` | summaries and figures |
| `hcp_rois_fsaverage5.npz` | atlas: region names and their vertex indices |
| `logs/` | full worker logs |
| `manifest.json` | package versions and settings |

**Handy tips**
- **Add more videos later:** add them to your dataset and Run all. Only new clips are processed.
- **Re-run a clip** (e.g. after changing `VARIANTS`): delete its folder in `tribe_outputs/clips/`.
- **Start completely fresh:** delete `/kaggle/working/tribe_runtime` and/or `tribe_outputs`.
- **Analysis only, no new clips:** the analysis cells need no GPU. You can also load the `.npy`/`.csv` files anywhere else.

**Interpreting results honestly:** these are model predictions for an "average" subject, not measured brain data. Check well-known contrasts first (faces → FFC, speech → STS/A1, motion → MT). Ablations replace a modality with zeros, which is an input the model rarely saw, so treat the RMS effects as relative rankings. Model weights are licensed CC BY-NC 4.0 (non-commercial use only).